# 03 — Modelling
**LSTM + Word2Vec  ·  BERT Fine-tuned Classifier**

This notebook trains two deep-learning models on features built in `02_feature_engineering`:

| Model | Input | Architecture |
|---|---|---|
| **LSTM + W2V** | Word2Vec token embeddings (300-d) | Bidirectional LSTM → Dense |
| **BERT Classifier** | Pre-computed BERT embeddings (384-d) | MLP head on frozen embeddings |

**Deliverables** (all saved to `MODELS_DIR`):
- Trained model weights (`.h5` / `.joblib`)
- Per-model metrics: Accuracy · Precision · Recall · F1-score
- Predictions arrays (`y_pred_*.npy`)
- Combined metrics summary CSV
- Confusion-matrix plots

## 1 · Imports & Config

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

# Deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

# Sklearn metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

# Gensim Word2Vec
from gensim.models import Word2Vec

print('TF version :', tf.__version__)
gpu = tf.config.list_physical_devices('GPU')
print('GPU available :', bool(gpu))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT   = '/content/drive/MyDrive/SmartReviewAnalyzer'
FEATURES_DIR = os.path.join(DRIVE_ROOT, 'data/features')
PROCESSED_DIR= os.path.join(DRIVE_ROOT, 'data/processed')
MODELS_DIR   = os.path.join(DRIVE_ROOT, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

# ── Hyper-parameters ──────────────────────────────────────────────
LSTM_MAXLEN      = 256     # pad/truncate sequences to this length
LSTM_EMBED_DIM   = 300     # must match W2V_VECTOR_SIZE in notebook 02
LSTM_UNITS       = 128     # BiLSTM hidden units (per direction)
LSTM_DROPOUT     = 0.3
LSTM_EPOCHS      = 10
LSTM_BATCH_SIZE  = 64

BERT_HIDDEN_UNITS= [256, 128]  # MLP layers on top of BERT embeddings
BERT_DROPOUT     = 0.3
BERT_EPOCHS      = 20
BERT_BATCH_SIZE  = 64

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

print('Config ready.')
print(f'  Features dir : {FEATURES_DIR}')
print(f'  Models dir   : {MODELS_DIR}')


## 2 · Load Features & Labels

In [ ]:
# ── Labels ──
labels = np.load(os.path.join(FEATURES_DIR, 'labels.npz'))
y_train = labels['y_train']
y_test  = labels['y_test']
print(f'y_train : {y_train.shape}  classes={np.unique(y_train)}')
print(f'y_test  : {y_test.shape}')

# ── BERT embeddings (pre-computed in notebook 02) ──
X_train_bert = np.load(os.path.join(FEATURES_DIR, 'X_train_bert.npy'))
X_test_bert  = np.load(os.path.join(FEATURES_DIR, 'X_test_bert.npy'))
print(f'BERT train : {X_train_bert.shape}')
print(f'BERT test  : {X_test_bert.shape}')

# ── Word2Vec model & raw text (for sequence encoding) ──
w2v_model = Word2Vec.load(os.path.join(FEATURES_DIR, 'word2vec.model'))
print(f'W2V vocab  : {len(w2v_model.wv):,} words, dim={w2v_model.wv.vector_size}')

# ── Raw preprocessed text for LSTM token sequences ──
df_train = pd.read_csv(os.path.join(PROCESSED_DIR, 'preprocessed_train.csv'))
df_test  = pd.read_csv(os.path.join(PROCESSED_DIR, 'preprocessed_test.csv'))
train_texts = df_train['cleaned_text_tfidf'].fillna('').tolist()
test_texts  = df_test['cleaned_text_tfidf'].fillna('').tolist()
print(f'Raw train texts : {len(train_texts):,}')


---
## 3 · LSTM + Word2Vec

Pipeline:
1. Build a **vocabulary index** from the W2V model
2. Convert each review to an **integer sequence**, pad to `LSTM_MAXLEN`
3. Initialise the **Embedding layer** with frozen W2V vectors
4. Stack a **Bidirectional LSTM** → Dropout → Dense(1, sigmoid)

### 3.1 · Build Vocabulary & Embedding Matrix

In [ ]:
wv = w2v_model.wv
vocab = wv.index_to_key          # list of words ordered by frequency
word2idx = {w: i+1 for i, w in enumerate(vocab)}  # 0 reserved for padding
vocab_size = len(vocab) + 1

# Embedding matrix: row i → vector for word with index i
embedding_matrix = np.zeros((vocab_size, LSTM_EMBED_DIM), dtype=np.float32)
for word, idx in word2idx.items():
    embedding_matrix[idx] = wv[word]

print(f'Vocabulary size (incl. PAD) : {vocab_size:,}')
print(f'Embedding matrix shape      : {embedding_matrix.shape}')


### 3.2 · Tokenise & Pad Sequences

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

def texts_to_sequences(texts, word2idx, maxlen):
    seqs = [[word2idx[w] for w in t.split() if w in word2idx]
             for t in texts]
    return pad_sequences(seqs, maxlen=maxlen, padding='post', truncating='post')

X_train_seq = texts_to_sequences(train_texts, word2idx, LSTM_MAXLEN)
X_test_seq  = texts_to_sequences(test_texts,  word2idx, LSTM_MAXLEN)

print(f'X_train_seq : {X_train_seq.shape}')
print(f'X_test_seq  : {X_test_seq.shape}')
print(f'Example     : {X_train_seq[0][:15]}')


### 3.3 · Build & Train Bi-LSTM Model

In [ ]:
def build_lstm_model(vocab_size, embed_dim, embedding_matrix, maxlen,
                     lstm_units=128, dropout=0.3):
    inp = keras.Input(shape=(maxlen,), name='token_ids')
    x   = layers.Embedding(
              input_dim    = vocab_size,
              output_dim   = embed_dim,
              weights      = [embedding_matrix],
              input_length = maxlen,
              trainable    = False,   # keep W2V vectors frozen
              name         = 'w2v_embedding'
          )(inp)
    x   = layers.Bidirectional(
              layers.LSTM(lstm_units, return_sequences=True), name='bilstm_1'
          )(x)
    x   = layers.Bidirectional(
              layers.LSTM(lstm_units // 2), name='bilstm_2'
          )(x)
    x   = layers.Dropout(dropout)(x)
    x   = layers.Dense(64, activation='relu')(x)
    x   = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation='sigmoid', name='output')(x)
    model = keras.Model(inp, out)
    return model

lstm_model = build_lstm_model(
    vocab_size, LSTM_EMBED_DIM, embedding_matrix, LSTM_MAXLEN,
    lstm_units=LSTM_UNITS, dropout=LSTM_DROPOUT
)
lstm_model.compile(
    optimizer = keras.optimizers.Adam(1e-3),
    loss      = 'binary_crossentropy',
    metrics   = ['accuracy']
)
lstm_model.summary()


In [ ]:
lstm_ckpt_path = os.path.join(MODELS_DIR, 'lstm_w2v_best.h5')

lstm_callbacks = [
    callbacks.ModelCheckpoint(
        lstm_ckpt_path, monitor='val_accuracy',
        save_best_only=True, verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_loss', patience=3,
        restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=2, min_lr=1e-6, verbose=1
    ),
]

print('Training LSTM + Word2Vec ...')
lstm_history = lstm_model.fit(
    X_train_seq, y_train,
    validation_split = 0.1,
    epochs           = LSTM_EPOCHS,
    batch_size       = LSTM_BATCH_SIZE,
    callbacks        = lstm_callbacks,
    verbose          = 1,
)


### 3.4 · Evaluate LSTM

In [ ]:
# Load best checkpoint weights
lstm_model.load_weights(lstm_ckpt_path)

lstm_proba  = lstm_model.predict(X_test_seq, batch_size=LSTM_BATCH_SIZE, verbose=0).ravel()
y_pred_lstm = (lstm_proba >= 0.5).astype(int)

np.save(os.path.join(MODELS_DIR, 'y_pred_lstm.npy'), y_pred_lstm)
np.save(os.path.join(MODELS_DIR, 'y_proba_lstm.npy'), lstm_proba)

lstm_metrics = dict(
    model     = 'LSTM + Word2Vec',
    accuracy  = accuracy_score(y_test, y_pred_lstm),
    precision = precision_score(y_test, y_pred_lstm, average='binary'),
    recall    = recall_score(y_test, y_pred_lstm, average='binary'),
    f1        = f1_score(y_test, y_pred_lstm, average='binary'),
)

print('\n── LSTM + Word2Vec ─────────────────────────────')
for k, v in lstm_metrics.items():
    if k != 'model': print(f'  {k:<12s}: {v:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_lstm, target_names=['Negative','Positive']))


---
## 4 · BERT Embedding Classifier

The pre-computed 384-d `all-MiniLM-L6-v2` embeddings are already L2-normalised.
We train a lightweight **MLP head** directly on top — no GPU needed for this step.

### 4.1 · Build & Train MLP on BERT Embeddings

In [ ]:
def build_bert_mlp(input_dim, hidden_units, dropout):
    inp = keras.Input(shape=(input_dim,), name='bert_embedding')
    x   = inp
    for units in hidden_units:
        x = layers.Dense(units, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation='sigmoid', name='output')(x)
    return keras.Model(inp, out)

bert_clf = build_bert_mlp(
    input_dim    = X_train_bert.shape[1],
    hidden_units = BERT_HIDDEN_UNITS,
    dropout      = BERT_DROPOUT
)
bert_clf.compile(
    optimizer = keras.optimizers.Adam(1e-3),
    loss      = 'binary_crossentropy',
    metrics   = ['accuracy']
)
bert_clf.summary()


In [ ]:
bert_ckpt_path = os.path.join(MODELS_DIR, 'bert_mlp_best.h5')

bert_callbacks = [
    callbacks.ModelCheckpoint(
        bert_ckpt_path, monitor='val_accuracy',
        save_best_only=True, verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_loss', patience=5,
        restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=3, min_lr=1e-6, verbose=1
    ),
]

print('Training BERT MLP classifier ...')
bert_history = bert_clf.fit(
    X_train_bert, y_train,
    validation_split = 0.1,
    epochs           = BERT_EPOCHS,
    batch_size       = BERT_BATCH_SIZE,
    callbacks        = bert_callbacks,
    verbose          = 1,
)


### 4.2 · Evaluate BERT Classifier

In [ ]:
bert_clf.load_weights(bert_ckpt_path)

bert_proba  = bert_clf.predict(X_test_bert, batch_size=BERT_BATCH_SIZE, verbose=0).ravel()
y_pred_bert = (bert_proba >= 0.5).astype(int)

np.save(os.path.join(MODELS_DIR, 'y_pred_bert.npy'), y_pred_bert)
np.save(os.path.join(MODELS_DIR, 'y_proba_bert.npy'), bert_proba)

bert_metrics = dict(
    model     = 'BERT MLP Classifier',
    accuracy  = accuracy_score(y_test, y_pred_bert),
    precision = precision_score(y_test, y_pred_bert, average='binary'),
    recall    = recall_score(y_test, y_pred_bert, average='binary'),
    f1        = f1_score(y_test, y_pred_bert, average='binary'),
)

print('\n── BERT MLP Classifier ─────────────────────────')
for k, v in bert_metrics.items():
    if k != 'model': print(f'  {k:<12s}: {v:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_bert, target_names=['Negative','Positive']))


---
## 5 · Model Comparison & Visualisations

### 5.1 · Metrics Summary Table

In [ ]:
all_metrics = [lstm_metrics, bert_metrics]
df_metrics  = pd.DataFrame(all_metrics).set_index('model')
df_metrics  = df_metrics.round(4)

print('\n══ Final Metrics ══════════════════════════════════')
print(df_metrics.to_string())

metrics_path = os.path.join(MODELS_DIR, 'metrics_summary.csv')
df_metrics.to_csv(metrics_path)
print(f'\nSaved → {metrics_path}')


### 5.2 · Bar Chart — All Metrics

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
df_metrics[['accuracy','precision','recall','f1']].plot(
    kind='bar', ax=ax, rot=15, width=0.65,
    color=['#4C72B0','#DD8452','#55A868','#C44E52']
)
ax.set_title('Model Comparison — Test-Set Metrics', fontsize=14, fontweight='bold')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right')
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=8, padding=2)
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'metrics_comparison.png'), dpi=150)
plt.show()


### 5.3 · Confusion Matrices

In [ ]:
def plot_confusion(y_true, y_pred, title, ax):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=ax,
        xticklabels=['Negative','Positive'],
        yticklabels=['Negative','Positive']
    )
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_confusion(y_test, y_pred_lstm, 'LSTM + Word2Vec',       axes[0])
plot_confusion(y_test, y_pred_bert, 'BERT MLP Classifier',   axes[1])
plt.suptitle('Confusion Matrices — Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'confusion_matrices.png'), dpi=150)
plt.show()


### 5.4 · Training Curves

In [ ]:
def plot_history(history, title, ax_acc, ax_loss):
    h = history.history
    epochs = range(1, len(h['accuracy'])+1)
    ax_acc.plot(epochs, h['accuracy'],     label='Train')
    ax_acc.plot(epochs, h['val_accuracy'], label='Val',  linestyle='--')
    ax_acc.set_title(f'{title} — Accuracy'); ax_acc.legend()
    ax_loss.plot(epochs, h['loss'],         label='Train')
    ax_loss.plot(epochs, h['val_loss'],     label='Val',  linestyle='--')
    ax_loss.set_title(f'{title} — Loss');   ax_loss.legend()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
plot_history(lstm_history, 'LSTM + W2V',   axes[0][0], axes[1][0])
plot_history(bert_history, 'BERT MLP',     axes[0][1], axes[1][1])
plt.suptitle('Training Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'training_curves.png'), dpi=150)
plt.show()


## 6 · Predictions — Deliverable

In [ ]:
# Combine all predictions into one DataFrame for easy inspection / downstream use
df_preds = pd.DataFrame({
    'y_true'          : y_test,
    'y_pred_lstm'     : y_pred_lstm,
    'y_proba_lstm'    : lstm_proba.round(4),
    'y_pred_bert'     : y_pred_bert,
    'y_proba_bert'    : bert_proba.round(4),
})

preds_path = os.path.join(MODELS_DIR, 'all_predictions.csv')
df_preds.to_csv(preds_path, index=False)

print('Predictions saved →', preds_path)
print(f'Shape : {df_preds.shape}')
df_preds.head(10)


## 7 · Deliverables Summary

In [ ]:
print('═'*55)
print(' DELIVERABLES')
print('═'*55)
artefacts = [
    ('Trained model (LSTM)',        'lstm_w2v_best.h5'),
    ('Trained model (BERT MLP)',    'bert_mlp_best.h5'),
    ('Metrics summary',             'metrics_summary.csv'),
    ('All predictions',             'all_predictions.csv'),
    ('Metric bar chart',            'metrics_comparison.png'),
    ('Confusion matrices',          'confusion_matrices.png'),
    ('Training curves',             'training_curves.png'),
    ('LSTM predictions (npy)',      'y_pred_lstm.npy'),
    ('BERT predictions (npy)',      'y_pred_bert.npy'),
]
for label, fname in artefacts:
    fpath = os.path.join(MODELS_DIR, fname)
    exists = os.path.exists(fpath)
    size   = f'{os.path.getsize(fpath)/1e6:.2f} MB' if exists else 'not found'
    status = '✓' if exists else '✗'
    print(f'  {status}  {label:<35s}  {fname:<35s}  {size}')
print('═'*55)
print('\nFinal Metrics:')
print(df_metrics.to_string())
